# Argentina vs England World Cup Semi-Final Prediction: xT/KMeans

This notebook mirrors the previous Spain-France tactical experiment for a new Argentina vs England semi-final prediction on 2026-07-15. It uses the local xT/KMeans outputs and formation summaries already produced in this project.

The Spain-France result note is treated as user-provided feedback. It lightly reinforces the factors that were useful last time: aggregate xT strength and formation matchup. It does not scrape current data or add the match as a training label.

In [ ]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs" / "argentina_england_prediction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEAM_A = "Argentina"
TEAM_B = "England"
MATCH_LABEL = "Argentina vs England"
MATCH_DATE = "2026-07-15"
POST_MATCH_FEEDBACK = "User-provided note: the previous Spain-France prediction was directionally right, with Spain beating France 2-0."

PRIMARY_CLUSTER_FILE = ROOT / "outputs" / "experiment_C_multichannel" / "clustered_team_seasons_multichannel.csv"
RAW_CLUSTER_FILE = ROOT / "outputs" / "experiment_A_raw" / "clustered_team_seasons_raw.csv"
SMOOTHED_CLUSTER_FILE = ROOT / "outputs" / "experiment_B_smoothed" / "clustered_team_seasons_smoothed.csv"
FORMATION_ROWS_FILE = ROOT / "outputs" / "formation_match_rows.csv"
FORMATION_SUMMARY_FILE = ROOT / "outputs" / "formation_matchup_summary.csv"
SPAIN_FRANCE_FINAL_FILE = ROOT / "outputs" / "spain_france_prediction" / "final_spain_france_prediction.csv"
SPAIN_FRANCE_STYLE_FILE = ROOT / "outputs" / "spain_france_prediction" / "spain_france_style_difference_vectors.csv"

ALLOWED_MENS_TOURNAMENTS = {"FIFA World Cup", "UEFA Euro", "Copa America"}
LESSON_REINFORCED_ZONES = ["received_z095", "created_z045", "created_z030"]
BASE_COMPONENT_WEIGHTS = {"strength": 0.55, "formation": 0.30, "style": 0.15}
REINFORCED_COMPONENT_WEIGHTS = {"strength": 0.60, "formation": 0.35, "style": 0.05}


def season_year(value):
    text = str(value)
    digits = "".join(ch for ch in text if ch.isdigit())
    return int(digits[:4]) if len(digits) >= 4 else -1


def normalize_probs(row, columns):
    total = float(sum(row[c] for c in columns))
    if total <= 0:
        for c in columns:
            row[c] = 1.0 / len(columns)
    else:
        for c in columns:
            row[c] = float(row[c]) / total
    return row


def sigmoid(x):
    return 1.0 / (1.0 + math.exp(-float(np.clip(x, -20, 20))))


def formation_to_label(value):
    text = str(value).replace(".0", "")
    if not text or text.lower() == "nan":
        return "unknown"
    if len(text) == 3:
        return "-".join(text)
    if len(text) == 4:
        return f"{text[0]}-{text[1]}-{text[2]}-{text[3]}"
    return text


def read_optional_csv(path):
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame()


inventory = []
for path in [PRIMARY_CLUSTER_FILE, RAW_CLUSTER_FILE, SMOOTHED_CLUSTER_FILE, FORMATION_ROWS_FILE, FORMATION_SUMMARY_FILE, SPAIN_FRANCE_FINAL_FILE, SPAIN_FRANCE_STYLE_FILE]:
    inventory.append({"path": str(path), "exists": path.exists(), "size_bytes": path.stat().st_size if path.exists() else 0})
pd.DataFrame(inventory).to_csv(OUTPUT_DIR / "project_file_inventory.csv", index=False)
pd.DataFrame(inventory)


## Load clustered xT rows

For this cross-confederation matchup, the notebook uses the latest available senior men's local tournament row for each team. Argentina can use Copa America rows while England can use Euro rows, because both are local senior tournament proxies.

In [ ]:

cluster_df = pd.read_csv(PRIMARY_CLUSTER_FILE)
cluster_col = [c for c in cluster_df.columns if c.startswith("cluster_")][-1]

mens_cluster_df = cluster_df[
    cluster_df["competition_name"].isin(ALLOWED_MENS_TOURNAMENTS)
    & ~cluster_df["team_name"].astype(str).str.contains("Women|Women's", case=False, na=False)
].copy()
mens_cluster_df["season_year"] = mens_cluster_df["season_name"].map(season_year)
mens_cluster_df["positive_xT_per_match"] = mens_cluster_df["total_positive_xT"] / mens_cluster_df["match_count"].replace(0, np.nan)
mens_cluster_df["net_xT_per_match"] = mens_cluster_df["total_net_xT"] / mens_cluster_df["match_count"].replace(0, np.nan)
mens_cluster_df["positive_pass_xT_per_match"] = mens_cluster_df["total_positive_pass_xT"] / mens_cluster_df["match_count"].replace(0, np.nan)
mens_cluster_df["positive_carry_xT_per_match"] = mens_cluster_df["total_positive_carry_xT"] / mens_cluster_df["match_count"].replace(0, np.nan)

def select_latest_team_row(team):
    rows = mens_cluster_df[mens_cluster_df["team_name"].eq(team)].copy()
    if rows.empty:
        raise ValueError(f"No clustered xT row found for {team}")
    rows = rows.sort_values(["season_year", "match_count", "positive_xT_per_match"], ascending=[False, False, False])
    return rows.iloc[0].copy()

team_a_row = select_latest_team_row(TEAM_A)
team_b_row = select_latest_team_row(TEAM_B)
target_profiles = pd.DataFrame([team_a_row, team_b_row])
target_profiles.to_csv(OUTPUT_DIR / "argentina_england_xt_profiles.csv", index=False)

assignments = target_profiles[[
    "team_name", "competition_name", "season_name", "match_count", cluster_col,
    "total_positive_xT", "positive_xT_per_match", "net_xT_per_match",
    "pass_xT_share", "carry_xT_share", "attacking_third_share", "center_attacking_third_share"
]].rename(columns={cluster_col: "cluster"})
assignments.to_csv(OUTPUT_DIR / "argentina_england_cluster_assignments.csv", index=False)
assignments


## xT style comparison

The model compares spatial xT profiles and also reports the same high-value xT zones that mattered in the Spain-France explanation.

In [ ]:

profile_cols = [c for c in mens_cluster_df.columns if c.startswith("created_z") or c.startswith("received_z")]
metric_cols = [
    "positive_xT_per_match", "net_xT_per_match", "positive_pass_xT_per_match",
    "positive_carry_xT_per_match", "pass_xT_share", "carry_xT_share",
    "attacking_third_share", "center_attacking_third_share",
]

a_vec = team_a_row[profile_cols].astype(float).fillna(0).to_numpy()
b_vec = team_b_row[profile_cols].astype(float).fillna(0).to_numpy()
cosine = float(np.dot(a_vec, b_vec) / ((np.linalg.norm(a_vec) * np.linalg.norm(b_vec)) or 1.0))
diff = a_vec - b_vec
style_edge = float(np.nanmean(diff) / (np.nanstd(mens_cluster_df[profile_cols].to_numpy(dtype=float)) or 1.0))

diff_vectors = pd.DataFrame({
    "feature": profile_cols,
    f"{TEAM_A}_value": team_a_row[profile_cols].astype(float).to_numpy(),
    f"{TEAM_B}_value": team_b_row[profile_cols].astype(float).to_numpy(),
})
diff_vectors["argentina_minus_england"] = diff_vectors[f"{TEAM_A}_value"] - diff_vectors[f"{TEAM_B}_value"]
diff_vectors["abs_difference"] = diff_vectors["argentina_minus_england"].abs()
diff_vectors = diff_vectors.sort_values("abs_difference", ascending=False)
diff_vectors.to_csv(OUTPUT_DIR / "argentina_england_style_difference_vectors.csv", index=False)

style_comparison_rows = [
    {"metric": "cosine_similarity_xt_surface", "value": cosine, "interpretation": "1 means very similar xT territory profile"},
    {"metric": "euclidean_distance_xt_surface", "value": float(np.linalg.norm(diff)), "interpretation": "larger means more different spatial profile"},
    {"metric": "manhattan_distance_xt_surface", "value": float(np.abs(diff).sum()), "interpretation": "larger means more total zone difference"},
]
for zone in LESSON_REINFORCED_ZONES:
    if zone in target_profiles.columns:
        style_comparison_rows.append({
            "metric": f"feedback_zone_edge_{zone}",
            "value": float(team_a_row[zone]) - float(team_b_row[zone]),
            "interpretation": "zone highlighted by the previous Spain-France xT explanation; positive favors Argentina here",
        })
style_comparison = pd.DataFrame(style_comparison_rows)
style_comparison.to_csv(OUTPUT_DIR / "argentina_england_style_comparison.csv", index=False)

GRID_L = 16
GRID_W = 12
N_ZONES = GRID_L * GRID_W

def ordered_zone_columns(columns, prefix):
    pairs = []
    for col in columns:
        if col.startswith(prefix):
            pairs.append((int(col.replace(prefix, "")), col))
    pairs = sorted(pairs)
    ordered = [col for _, col in pairs]
    if len(ordered) != N_ZONES:
        raise ValueError(f"Expected {N_ZONES} {prefix} columns for a {GRID_W} by {GRID_L} pitch grid, found {len(ordered)}")
    return ordered

def zone_grid(row, columns):
    values = row[columns].astype(float).to_numpy()
    return values.reshape(GRID_W, GRID_L)

def plot_pitch_grid(grid, title, output_path, cmap="viridis", vmin=None, vmax=None, colorbar_label="created xT share"):
    fig, ax = plt.subplots(figsize=(10, 6))
    image = ax.imshow(grid, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax, extent=[0, GRID_L, 0, GRID_W])
    ax.set_title(title)
    ax.set_xlabel("Pitch length zones")
    ax.set_ylabel("Pitch width zones")
    ax.set_xticks(np.arange(0, GRID_L + 1, 1), minor=True)
    ax.set_yticks(np.arange(0, GRID_W + 1, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.45, alpha=0.75)
    ax.set_xticks([0, 4, 8, 12, 16])
    ax.set_yticks([0, 3, 6, 9, 12])
    ax.set_xlim(0, GRID_L)
    ax.set_ylim(0, GRID_W)
    fig.colorbar(image, ax=ax, label=colorbar_label, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.close(fig)

created_cols = ordered_zone_columns(profile_cols, "created_z")
for label, row in [(TEAM_A, team_a_row), (TEAM_B, team_b_row)]:
    grid = zone_grid(row, created_cols)
    plot_pitch_grid(grid, f"{label} primary created xT profile", OUTPUT_DIR / f"{label.lower()}_primary_xt_heatmap.png")

vals = team_a_row[created_cols].astype(float).to_numpy() - team_b_row[created_cols].astype(float).to_numpy()
grid = vals.reshape(GRID_W, GRID_L)
lim = max(abs(float(np.nanmin(grid))), abs(float(np.nanmax(grid)))) or 1
plot_pitch_grid(
    grid,
    "Created xT difference",
    OUTPUT_DIR / "argentina_minus_england_xt_difference.png",
    cmap="coolwarm",
    vmin=-lim,
    vmax=lim,
    colorbar_label="Argentina minus England created xT",
)

diff_vectors.head(12)


## Strength prior

The strength prior is based on xT volume, net xT, pass/carry contribution, and attacking-third occupation relative to the local senior men's pool.

In [ ]:

# Strength prior from the local clustered-team pool.
strength_metrics = ["positive_xT_per_match", "net_xT_per_match", "positive_pass_xT_per_match", "positive_carry_xT_per_match", "attacking_third_share", "center_attacking_third_share"]
weights = {
    "positive_xT_per_match": 0.34,
    "net_xT_per_match": 0.24,
    "positive_pass_xT_per_match": 0.14,
    "positive_carry_xT_per_match": 0.12,
    "attacking_third_share": 0.08,
    "center_attacking_third_share": 0.08,
}
pool = mens_cluster_df[strength_metrics].astype(float)
means = pool.mean()
stds = pool.std().replace(0, 1)

def strength_score(row):
    score = 0.0
    parts = []
    for metric, weight in weights.items():
        z = float((float(row[metric]) - means[metric]) / stds[metric])
        score += weight * z
        parts.append({"metric": metric, "z_score": z, "weight": weight, "weighted_value": weight * z})
    return score, pd.DataFrame(parts)

a_strength, a_parts = strength_score(team_a_row)
b_strength, b_parts = strength_score(team_b_row)
strength_edge = a_strength - b_strength
draw_strength = float(np.clip(0.29 - 0.035 * abs(strength_edge), 0.20, 0.32))
a_non_draw = sigmoid(strength_edge)
strength_probs = {
    "component": "strength_prior",
    "argentina_win_90": (1 - draw_strength) * a_non_draw,
    "draw_90": draw_strength,
    "england_win_90": (1 - draw_strength) * (1 - a_non_draw),
    "edge": strength_edge,
}
strength_parts = pd.concat([
    a_parts.assign(team=TEAM_A),
    b_parts.assign(team=TEAM_B),
], ignore_index=True)
strength_summary = pd.DataFrame([
    {"team": TEAM_A, "strength_score": a_strength, **{m: float(team_a_row[m]) for m in strength_metrics}},
    {"team": TEAM_B, "strength_score": b_strength, **{m: float(team_b_row[m]) for m in strength_metrics}},
])
strength_summary.to_csv(OUTPUT_DIR / "argentina_england_strength_prior.csv", index=False)
strength_parts.to_csv(OUTPUT_DIR / "argentina_england_strength_components.csv", index=False)
strength_summary


## Formation scenarios

Formation scenarios are estimated from local formation match rows. This is a transparent tactical proxy, not a lineup-aware model.

In [ ]:

formation_rows = read_optional_csv(FORMATION_ROWS_FILE)
if formation_rows.empty:
    raise ValueError("formation_match_rows.csv is required for the tactical scenario section")

formation_rows = formation_rows[~formation_rows["team"].astype(str).str.contains("Women|Women's", case=False, na=False)].copy()
formation_rows["formation_label"] = formation_rows["formation"].map(formation_to_label)
formation_rows["opponent_formation_label"] = formation_rows["opponent_formation"].map(formation_to_label)

def top_formations(team, fallback):
    rows = formation_rows[formation_rows["team"].eq(team)].copy()
    if rows.empty:
        return pd.DataFrame({"formation_label": fallback, "usage": [1 / len(fallback)] * len(fallback), "points_per_game": [1.33] * len(fallback), "goal_diff_per_game": [0.0] * len(fallback)})
    grouped = rows.groupby("formation_label").agg(
        games=("match_id", "count"),
        points_per_game=("points", "mean"),
        goal_diff_per_game=("goal_diff", "mean"),
        win_rate=("result", lambda s: (s == "W").mean()),
    ).reset_index()
    grouped["usage"] = grouped["games"] / grouped["games"].sum()
    return grouped.sort_values(["usage", "points_per_game"], ascending=False).head(3)

a_forms = top_formations(TEAM_A, ["4-3-3", "4-4-1-1", "3-5-2"])
b_forms = top_formations(TEAM_B, ["4-2-3-1", "4-3-3", "3-4-2-1"])

scenarios = []
for _, af in a_forms.iterrows():
    for _, bf in b_forms.iterrows():
        a_form_score = (float(af["points_per_game"]) - 1.33) / 1.67 + 0.18 * float(af["goal_diff_per_game"])
        b_form_score = (float(bf["points_per_game"]) - 1.33) / 1.67 + 0.18 * float(bf["goal_diff_per_game"])
        edge = a_form_score - b_form_score
        usage_weight = float(af["usage"]) * float(bf["usage"])
        draw = float(np.clip(0.30 - 0.03 * abs(edge), 0.22, 0.33))
        p_a = sigmoid(edge)
        row = {
            "argentina_formation": af["formation_label"],
            "england_formation": bf["formation_label"],
            "scenario_weight": usage_weight,
            "formation_edge": edge,
            "argentina_win_90": (1 - draw) * p_a,
            "draw_90": draw,
            "england_win_90": (1 - draw) * (1 - p_a),
            "argentina_ppg_in_shape": float(af["points_per_game"]),
            "england_ppg_in_shape": float(bf["points_per_game"]),
        }
        scenarios.append(normalize_probs(row, ["argentina_win_90", "draw_90", "england_win_90"]))

scenario_df = pd.DataFrame(scenarios)
scenario_df["scenario_weight"] = scenario_df["scenario_weight"] / scenario_df["scenario_weight"].sum()
scenario_df.to_csv(OUTPUT_DIR / "argentina_england_formation_scenarios.csv", index=False)

formation_probs = {
    "component": "formation_scenarios",
    "argentina_win_90": float((scenario_df["argentina_win_90"] * scenario_df["scenario_weight"]).sum()),
    "draw_90": float((scenario_df["draw_90"] * scenario_df["scenario_weight"]).sum()),
    "england_win_90": float((scenario_df["england_win_90"] * scenario_df["scenario_weight"]).sum()),
    "edge": float((scenario_df["formation_edge"] * scenario_df["scenario_weight"]).sum()),
}
formation_probs = normalize_probs(formation_probs, ["argentina_win_90", "draw_90", "england_win_90"])

plt.figure(figsize=(9, 5))
plot_df = scenario_df.sort_values("scenario_weight", ascending=False).head(8).copy()
plot_df["scenario"] = plot_df["argentina_formation"] + " vs " + plot_df["england_formation"]
plt.barh(plot_df["scenario"], plot_df["argentina_win_90"], label="Argentina")
plt.barh(plot_df["scenario"], plot_df["draw_90"], left=plot_df["argentina_win_90"], label="Draw")
plt.barh(plot_df["scenario"], plot_df["england_win_90"], left=plot_df["argentina_win_90"] + plot_df["draw_90"], label="England")
plt.xlabel("90-minute probability")
plt.title("Formation scenario probabilities")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "argentina_england_formation_scenarios.png", dpi=180)
plt.close()

scenario_df.sort_values("scenario_weight", ascending=False).head(10)


## Reinforced final prediction

The final table compares the baseline weights against a lightly reinforced version after the user-provided Spain-France feedback.

In [ ]:

# Style component with a small feedback-zone diagnostic from the Spain-France explanation.
feedback_zone_edge = 0.0
feedback_zone_count = 0
for zone in LESSON_REINFORCED_ZONES:
    if zone in target_profiles.columns:
        scale = float(mens_cluster_df[zone].std()) or 1.0
        feedback_zone_edge += (float(team_a_row[zone]) - float(team_b_row[zone])) / scale
        feedback_zone_count += 1
if feedback_zone_count:
    feedback_zone_edge /= feedback_zone_count

style_edge = 0.65 * strength_edge + 0.35 * feedback_zone_edge
draw_style = float(np.clip(0.31 - 0.02 * abs(style_edge), 0.24, 0.34))
style_a = sigmoid(style_edge)
style_probs = {
    "component": "xt_style_feedback_zones",
    "argentina_win_90": (1 - draw_style) * style_a,
    "draw_90": draw_style,
    "england_win_90": (1 - draw_style) * (1 - style_a),
    "edge": style_edge,
    "feedback_zone_edge": feedback_zone_edge,
}
style_probs = normalize_probs(style_probs, ["argentina_win_90", "draw_90", "england_win_90"])

component_df = pd.DataFrame([strength_probs, formation_probs, style_probs])
component_df.to_csv(OUTPUT_DIR / "argentina_england_tactical_scenario_predictions.csv", index=False)

def combine_components(weight_map):
    out = {"weighting": "custom"}
    for prob_col in ["argentina_win_90", "draw_90", "england_win_90"]:
        out[prob_col] = 0.0
    for _, row in component_df.iterrows():
        component = row["component"]
        if component == "strength_prior":
            w = weight_map["strength"]
        elif component == "formation_scenarios":
            w = weight_map["formation"]
        else:
            w = weight_map["style"]
        for prob_col in ["argentina_win_90", "draw_90", "england_win_90"]:
            out[prob_col] += w * float(row[prob_col])
    return normalize_probs(out, ["argentina_win_90", "draw_90", "england_win_90"])

baseline = combine_components(BASE_COMPONENT_WEIGHTS)
baseline["weighting"] = "baseline_before_feedback"
reinforced = combine_components(REINFORCED_COMPONENT_WEIGHTS)
reinforced["weighting"] = "reinforced_after_user_provided_spain_france_feedback"

final_df = pd.DataFrame([baseline, reinforced])
final_df["predicted_90_minute_result"] = final_df[["argentina_win_90", "draw_90", "england_win_90"]].idxmax(axis=1).map({
    "argentina_win_90": "Argentina win",
    "draw_90": "Draw",
    "england_win_90": "England win",
})
final_df["match"] = MATCH_LABEL
final_df["match_date"] = MATCH_DATE
final_df.to_csv(OUTPUT_DIR / "final_argentina_england_prediction.csv", index=False)

uncertainty = pd.DataFrame([
    {"item": "post_match_feedback", "value": POST_MATCH_FEEDBACK},
    {"item": "reinforced_factor", "value": "Strength prior and formation matchup receive slightly more weight after the previous Spain-France prediction was directionally correct."},
    {"item": "overfit_guardrail", "value": "The single Spain-France result is used only as a light interpretation/weighting update, not as a new training row."},
    {"item": "data_scope", "value": "Local senior men's xT clusters; Argentina can draw from Copa America rows and England from Euro rows."},
])
uncertainty.to_csv(OUTPUT_DIR / "prediction_uncertainty.csv", index=False)

plt.figure(figsize=(7, 4))
reinforced_plot = final_df[final_df["weighting"].str.startswith("reinforced")].iloc[0]
labels = ["Argentina win", "Draw", "England win"]
vals = [reinforced_plot["argentina_win_90"], reinforced_plot["draw_90"], reinforced_plot["england_win_90"]]
plt.bar(labels, vals, color=["#4c78a8", "#9e9e9e", "#f58518"])
plt.ylim(0, max(vals) * 1.25)
plt.ylabel("Probability")
plt.title("Argentina vs England xT/KMeans tactical prediction")
for i, v in enumerate(vals):
    plt.text(i, v + 0.01, f"{v:.1%}", ha="center")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "argentina_england_final_probabilities.png", dpi=180)
plt.close()

lesson_df = pd.DataFrame([
    {"factor": "team_strength_prior", "before_weight": BASE_COMPONENT_WEIGHTS["strength"], "after_weight": REINFORCED_COMPONENT_WEIGHTS["strength"], "reason": "Spain-France prediction leaned correctly toward Spain through the aggregate strength signal."},
    {"factor": "formation_matchup", "before_weight": BASE_COMPONENT_WEIGHTS["formation"], "after_weight": REINFORCED_COMPONENT_WEIGHTS["formation"], "reason": "Spain-France explanation used formation/scenario structure as a material factor."},
    {"factor": "xT_feedback_zones", "before_weight": BASE_COMPONENT_WEIGHTS["style"], "after_weight": REINFORCED_COMPONENT_WEIGHTS["style"], "reason": "The same zones are retained as diagnostics, but not over-weighted from one match."},
])
lesson_df.to_csv(OUTPUT_DIR / "argentina_england_reinforcement_notes.csv", index=False)

report = f"""# Argentina vs England xT/KMeans Prediction

Match date: {MATCH_DATE}

This notebook is a companion to the Spain-France tactical prediction notebook. It uses local xT/KMeans team-season outputs and local formation summaries only. It does not use current 2026 event data, lineups, odds, or media commentary.

## Spain-France feedback

{POST_MATCH_FEEDBACK}

I treat that as a light model-calibration lesson rather than a new supervised label. The reinforced version gives a little more weight to the two factors that were most useful in the previous explanation: aggregate xT/team-strength and formation matchup. The xT zones highlighted last time (`received_z095`, `created_z045`, `created_z030`) are kept as diagnostics.

## Main reinforced prediction

- Argentina win in 90 minutes: {reinforced_plot['argentina_win_90']:.1%}
- Draw after 90 minutes: {reinforced_plot['draw_90']:.1%}
- England win in 90 minutes: {reinforced_plot['england_win_90']:.1%}
- Most likely 90-minute result bucket: {reinforced_plot['predicted_90_minute_result']}

## Data notes

Argentina selected row: {team_a_row['competition_name']} {team_a_row['season_name']} with {int(team_a_row['match_count'])} matches.
England selected row: {team_b_row['competition_name']} {team_b_row['season_name']} with {int(team_b_row['match_count'])} matches.

Because this is a cross-confederation match, the local proxy data naturally compares Argentina's latest available senior men's tournament row against England's latest available senior men's tournament row. This is useful for an educational experiment, but it is not betting-grade.
"""
(OUTPUT_DIR / "argentina_england_prediction_report.md").write_text(report, encoding="utf-8")

checklist = pd.DataFrame([
    {"check": "output_folder_created", "status": OUTPUT_DIR.exists()},
    {"check": "primary_cluster_file_loaded", "status": PRIMARY_CLUSTER_FILE.exists()},
    {"check": "formation_rows_loaded", "status": FORMATION_ROWS_FILE.exists()},
    {"check": "final_probabilities_sum_to_one", "status": bool(np.allclose(final_df[["argentina_win_90", "draw_90", "england_win_90"]].sum(axis=1), 1.0))},
    {"check": "spain_france_feedback_written", "status": True},
])
checklist.to_csv(OUTPUT_DIR / "final_checklist.csv", index=False)

print("Completed xT/KMeans Argentina vs England notebook.")
print("Output folder:", OUTPUT_DIR)
print(final_df.to_string(index=False))
